## num_warps: How Many Warps Run One Triton Program

`num_warps` tells Triton how many GPU warps to use for one program instance. A CUDA warp has 32 lanes, so `num_warps=4` means the program is scheduled with 128 lanes worth of execution resources.

This is a compile-time launch/config parameter, not a normal tensor argument. Triton will compile a different kernel variant for different `num_warps` values.

Reference: [triton.Config](https://triton-lang.org/main/python-api/generated/triton.Config.html)

## Mental Model

One Triton program is similar to one CUDA thread block / CTA. The launch grid decides how many programs run. `num_warps` decides how much warp-level parallelism each program gets.

- Small tiles often work well with `1` or `2` warps.
- Medium matmul tiles often work well with `4` warps.
- Large tiles may need `8` warps to expose enough work.
- Too many warps can reduce occupancy or add scheduling overhead.

There is no globally best value. Treat it as a performance knob and benchmark it.

In [ ]:
import torch
import triton
import triton.language as tl
from triton.testing import do_bench

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU."

## A Matmul Kernel

We will keep the tile sizes fixed and sweep only `num_warps`. That isolates the effect of the warp count.

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offsets_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offsets_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offsets_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = a_ptr + offsets_m[:, None] * stride_am + offsets_k[None, :] * stride_ak
    b_ptrs = b_ptr + offsets_k[:, None] * stride_bk + offsets_n[None, :] * stride_bn
    c_ptrs = c_ptr + offsets_m[:, None] * stride_cm + offsets_n[None, :] * stride_cn

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    for k in range(0, K, BLOCK_SIZE_K):
        a_mask = (offsets_m[:, None] < M) & (offsets_k[None, :] + k < K)
        b_mask = (offsets_k[:, None] + k < K) & (offsets_n[None, :] < N)

        a = tl.load(a_ptrs + k * stride_ak, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs + k * stride_bk, mask=b_mask, other=0.0)
        accumulator += tl.dot(a, b)

    c_mask = (offsets_m[:, None] < M) & (offsets_n[None, :] < N)
    tl.store(c_ptrs, accumulator, mask=c_mask)

## Launcher With num_warps

`num_warps` is passed at launch time, next to the compile-time block sizes. It does not appear in the kernel signature.

In [ ]:
def matmul_with_warps(
    a: torch.Tensor,
    b: torch.Tensor,
    num_warps: int,
    block_m: int = 128,
    block_n: int = 128,
    block_k: int = 32,
) -> torch.Tensor:
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_cuda and b.is_cuda

    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)

    grid = lambda META: (
        triton.cdiv(M, META["BLOCK_SIZE_M"]),
        triton.cdiv(N, META["BLOCK_SIZE_N"]),
    )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=block_m,
        BLOCK_SIZE_N=block_n,
        BLOCK_SIZE_K=block_k,
        num_warps=num_warps,
        num_stages=3,
    )
    return c

## Benchmark Different Warp Counts

The first run of each value compiles a specialized kernel. The benchmark below does one correctness call first, then measures steady-state execution.

In [ ]:
def benchmark_num_warps(M: int = 1024, N: int = 1024, K: int = 1024):
    torch.manual_seed(0)
    a = torch.randn((M, K), device="cuda", dtype=torch.float16)
    b = torch.randn((K, N), device="cuda", dtype=torch.float16)

    expected = a @ b
    torch.cuda.synchronize()

    print(f"shape: A={tuple(a.shape)}, B={tuple(b.shape)}")
    print("fixed tile: BLOCK_M=128, BLOCK_N=128, BLOCK_K=32, num_stages=3")

    for num_warps in [1, 2, 4, 8]:
        c = matmul_with_warps(a, b, num_warps=num_warps)
        torch.cuda.synchronize()
        max_diff = torch.max(torch.abs(c - expected)).item()

        ms = do_bench(lambda: matmul_with_warps(a, b, num_warps=num_warps), warmup=25, rep=100)
        print(f"num_warps={num_warps:<2}  time={ms:.3f} ms  max_diff={max_diff:.3g}")


benchmark_num_warps()

## Reading the Results

If `num_warps=1` is slow, the tile probably has too much work for a single warp group to schedule efficiently. If `num_warps=8` is slow, each program may be using enough resources that fewer programs can stay resident on each SM.

For this tile size, `4` or `8` warps often wins, but the best value depends on GPU architecture, tile shape, dtype, and problem size.

## Using num_warps in Autotune

`num_warps` is usually tuned together with tile sizes. Larger tiles generally need more warps than smaller tiles.

```python
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE_M": 64,  "BLOCK_SIZE_N": 64,  "BLOCK_SIZE_K": 32}, num_warps=2, num_stages=3),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_warps=4, num_stages=3),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32}, num_warps=8, num_stages=3),
    ],
    key=["M", "N", "K"],
)
@triton.jit
def matmul_kernel(...):
    ...
```

Autotune benchmarks these candidates and keeps the fastest config for the current shape.

## Exercise

Change the tile to `BLOCK_M=64, BLOCK_N=64` and run the benchmark again. Then try `BLOCK_M=128, BLOCK_N=256`.

You should see why `num_warps` belongs in the autotune search space: the best value moves when the tile shape changes.